# The XAG Oracle and Belletti-Simple Are One Construction

The revision found that the oracle synthesized by Logique's `xag` method and the
`simple` oracle of the Saha–Belletti package (Belletti-Simple) need the same number
of T gates and qubits once Belletti-Simple's multi-controlled X gates are expanded
with the throwaway-ancilla tree of the manuscript's Figure 1. This notebook shows
why. Under the edge-only predicate of regime R3, both circuits compute the same AND
gates of the same signals and hold each intermediate AND on its own qubit. They
differ only in their free linear gates (CNOT and X), in how each conjunction is
grouped, and in the order of independent gates.

The sections cover:

* the two constructions and their analytical counts (Section 2);
* the triangle $K_3$ with three colors: both logical oracles, their expansion and
  the exact Clifford+T gates that are counted (Section 3);
* both expanded circuits read back as AND networks (Section 4);
* an exhaustive check that both oracles act identically (Section 5);
* a wider comparator ($c = 3$), where Figure 1's tree appears inside every
  comparator (Section 6);
* the nine benchmark graphs: the same AND count, T-count, qubits and formula
  (Section 7);
* where the two constructions differ: T-depth, width under restoring
  decompositions, and the domain checks of regime R2 (Section 8);
* what this means for the manuscript (Section 9).

Circuit wires are drawn as named registers. `v1`, `v2`, … hold the color bits of
each vertex, named after the vertex labels of the input file, least-significant bit
first; `w` is the XAG's AND workspace; `e` holds
Belletti-Simple's constraint qubit for each edge; `a` holds ancillas appended by an
MCX decomposition; and `out` is the phase kick-back qubit, prepared in
$|-\rangle$.

## 1. Set up the run

The notebook runner supplies `LOGIQUE_OUTPUT`; an interactive run creates a new
folder under `dump/`. Figures are saved as PNG and SVG files in that folder. When
`LOGIQUE_STUDY_RUN` names a saved study run, Section 7 also compares the numbers
computed here with that run.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from graphviz import Digraph
from IPython.display import SVG, Image, Markdown, display
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ControlledGate
from qiskit.quantum_info import Operator

from logique.benchmarks import costs
from logique.benchmarks.study import expand_oracle
from logique.circuits.search import baseline_instance, logique_instance
from logique.coloring import read_dimacs
from logique.paths import native_executable, new_dump_directory

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    OUT = new_dump_directory("xag-belletti-equivalence", root=ROOT / "dump")
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
EXECUTABLE = native_executable(
    os.environ.get("LOGIQUE_NATIVE_EXECUTABLE")
    or ROOT / "external/native/boolean_synthesis/build/boolean_synthesis"
)
GRAPHS = ROOT / "datasets/graphs/benchmarks"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)


def show_figure(figure, stem: str) -> None:
    """Save a Matplotlib figure as PNG and SVG, then display it."""
    figure.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    figure.savefig(FIGURES / f"{stem}.png", dpi=140, bbox_inches="tight")
    display(Image(filename=str(FIGURES / f"{stem}.png")))
    plt.close(figure)


def show_svg(svg: bytes, stem: str) -> None:
    """Save and display an SVG drawing produced by Graphviz."""
    (FIGURES / f"{stem}.svg").write_bytes(svg)
    display(SVG(data=svg))


BLOCK_LABELS = {"ccx": "Toffoli", "rccx": "RCCX", "rccx_dg": "RCCX†", "ccrx": "CC-Rx(±π)"}


def show_blockwise(circuit, stem: str, fold: int = 60) -> None:
    """Draw each block gate as its counted Clifford+T gadget between labelled barriers."""
    expanded = circuit.copy_empty_like()
    for index, inst in enumerate(circuit.data):
        label = BLOCK_LABELS.get(inst.operation.name.split("_o")[0])
        if label is None:
            expanded.append(inst.operation, inst.qubits, inst.clbits)
            continue
        block = QuantumCircuit(len(inst.qubits))
        block.append(inst.operation, range(len(inst.qubits)))
        expanded.barrier(label=label)
        expanded.compose(costs.clifford_t(block), inst.qubits, inplace=True)
        following = (
            circuit.data[index + 1].operation.name.split("_o")[0]
            if index + 1 < len(circuit.data)
            else None
        )
        if following not in BLOCK_LABELS:
            expanded.barrier()
    counts = costs.ft_metrics(costs.clifford_t(circuit))
    display(
        Markdown(
            f"Expanded block by block: {counts['t_count']} T gates, T-depth "
            f"{counts['t_depth']:.0f}, {counts['cx_count']} CNOTs."
        )
    )
    show_figure(expanded.draw("mpl", fold=fold, idle_wires=False), stem + "-blocks")


def role_groups(instance, circuit):
    """Wire groups in drawing order: one register per vertex, then each role."""
    groups = [
        (f"v{vertex}", [instance.data[i] for i in inputs])
        for vertex, inputs in instance.encoding.input_mapping.items()
    ]
    role = "ancillas" if instance.method.startswith("sb_") else "workspace"
    groups.append(("e" if role == "ancillas" else "w", instance.roles[role]))
    groups.append(("a", list(range(instance.oracle.num_qubits, circuit.num_qubits))))
    groups.append(("out", instance.roles["output"]))
    return [(name, wires) for name, wires in groups if wires]


def labelled(circuit, groups):
    """Copy a circuit onto named registers so that drawings show qubit roles."""
    registers = [QuantumRegister(len(wires), name) for name, wires in groups]
    drawn = QuantumCircuit(*registers, global_phase=circuit.global_phase)
    position = {}
    for register, (_, wires) in zip(registers, groups):
        position.update(zip(wires, register))
    if len(position) != circuit.num_qubits:
        raise ValueError("Every qubit must belong to exactly one group")
    for inst in circuit.data:
        wires = [position[circuit.find_bit(q).index] for q in inst.qubits]
        drawn.append(inst.operation, wires)
    return drawn


def wire_names(groups):
    return {q: f"{name}[{i}]" for name, wires in groups for i, q in enumerate(wires)}


def expanded(instance, policy: str = "throwaway_tree"):
    """Expand an oracle as the study does: MCX policy, then relative-phase pairs."""
    circuit, records = expand_oracle(instance, policy)
    return costs.paired_relative_phase_toffolis(circuit), records


print("Artifacts:", OUT)

## 2. Two constructions of the edge-only predicate

Each vertex $v$ holds a color code of $c = \lceil \log_2 k \rceil$ bits
$x_{v,0}, \dots, x_{v,c-1}$. Under regime R3 the preparation produces only the $k$
valid codes, so the oracle checks the edge inequalities alone:

$$f(x) = \bigwedge_{(u,v) \in E} \neg\,\mathrm{eq}_{uv}(x), \qquad
\mathrm{eq}_{uv}(x) = \bigwedge_{i < c} \neg\,(x_{u,i} \oplus x_{v,i}).$$

**XAG oracle (Logique).** The encoder writes $f$ as structural Verilog. Its XAG has
$c - 1$ AND gates per edge for $\mathrm{eq}_{uv}$ and $|E| - 1$ AND gates for the
conjunction. XOR and NOT gates are free: `xag_synth` (Meuli et al.) realizes them
with CNOT and X gates applied to the data qubits in place. Every AND gate except
the last receives a fresh workspace qubit and is computed with a doubly controlled
$R_x(\pi)$, a relative-phase Toffoli gate. The last AND gate writes into `out` with
an exact Toffoli gate, and the workspace is then uncomputed.

**Belletti-Simple.** One constraint qubit per edge is set to $|1\rangle$. The
comparator of an edge XORs the register of $u$ into that of $v$, negates it, and
flips the constraint qubit with a $c$-control X gate when the two colors agree, so
the qubit holds $\neg\,\mathrm{eq}_{uv}$; the register of $v$ is then restored. An
$|E|$-control X gate writes the conjunction into `out`, and the comparators are
undone in reverse order.

**Decomposition.** The throwaway tree expands an $n$-control X gate into $n - 1$
Toffoli gates and $n - 2$ fresh ancillas, which it leaves in a garbage state that
the mirrored comparator later removes. Each comparator therefore contributes
$c - 1$ Toffoli gates and $c - 2$ ancillas. The conjunction has no mirror, so the
study expands it with the restoring tree, which needs the same $|E| - 2$ ancillas
and $|E| - 1$ Toffoli gates. Inside the mirrored block every Toffoli gate may carry
a relative phase (4 T gates to compute and 4 to uncompute); only the gate on `out`
is exact (7 T gates).

| | XAG oracle | Belletti-Simple, throwaway tree |
|---|---|---|
| AND gates computed | $(c-1)\lvert E\rvert + \lvert E\rvert - 1$ | $(c-1)\lvert E\rvert + \lvert E\rvert - 1$ |
| Data qubits | $\lvert V\rvert c$ | $\lvert V\rvert c$ |
| Qubits holding an intermediate AND | $\lvert E\rvert c - 2$ in `w` | $\lvert E\rvert$ in `e`, $\lvert E\rvert(c-2) + \lvert E\rvert - 2$ in `a` |
| Total qubits | $(\lvert V\rvert + \lvert E\rvert)c - 1$ | $(\lvert V\rvert + \lvert E\rvert)c - 1$ |
| T-count | $8(\lvert E\rvert c - 2) + 7$ | $8(\lvert E\rvert c - 2) + 7$ |

The formulas assume $c \ge 2$ and $|E| \ge 2$. The cells below check every entry.

In [ ]:
def predicted(vertices: int, edges: int, bits: int) -> dict:
    """Analytical counts shared by both constructions under the throwaway tree."""
    ands = edges * bits - 1
    return {
        "AND gates": ands,
        # Every AND but the last is computed and uncomputed.
        "Toffoli-type gates": 2 * ands - 1,
        "qubits": (vertices + edges) * bits - 1,
        "T-count": 8 * (ands - 1) + 7,
    }

## 3. The triangle with three colors

$K_3$ with $k = 3$ colors gives $c = 2$ bits per vertex. Under R3 the code $11$ is
never prepared, so both oracles check the three edges only. Both instances are
built and verified by the library exactly as in the study: the raw XAG circuit
passes the kick-back check for both target values, and the baseline passes the
XOR-action check. The XAG below has $c - 1 = 1$ AND gate per edge and two more for
the conjunction.

In [ ]:
from logique.synthesis import synthesize
from logique.visualization.boolean import graphviz_graph

problem = replace(read_dimacs(GRAPHS / "K3.col"), precolored={})
xag = logique_instance(problem, "xag", "R3", executable=EXECUTABLE)
simple = baseline_instance(problem, "simple", "R3")
display(
    pd.DataFrame(
        {
            name: {
                "oracle form": instance.form,
                "qubits (logical)": instance.oracle.num_qubits,
                "validation": instance.validation["status"],
                "assignments checked": instance.validation["assignments"],
                "marked assignments": instance.validation["marked_assignments"],
            }
            for name, instance in (("XAG", xag), ("Belletti-Simple", simple))
        }
    )
)

input_labels = [None] * xag.encoding.input_count
for vertex, inputs in xag.encoding.input_mapping.items():
    for bit, i in enumerate(inputs):
        input_labels[i] = f"x_{vertex},{bit}"
network = synthesize(xag.encoding, "xag", executable=EXECUTABLE).metadata["network"]
dot = graphviz_graph(network, "XAG of the edge-only predicate, K3 (c = 2)", input_labels=input_labels)
show_svg(dot.pipe(format="svg"), "k3-xag")

### The logical oracles

Before decomposition, Belletti-Simple contains one Toffoli gate per comparator
(a comparator with $c = 2$ bits has two controls) and a three-control X gate for
the conjunction, while the XAG oracle already consists of two-control gates. The
`x` gates of Belletti-Simple set the constraint qubits to $|1\rangle$ and negate
the compared register; the XAG flow folds the same negations into open controls,
drawn as hollow circles.

In [ ]:
xag_groups = role_groups(xag, xag.oracle)
simple_groups = role_groups(simple, simple.oracle)
show_figure(labelled(xag.oracle, xag_groups).draw("mpl", fold=40), "k3-xag-logical")
show_figure(labelled(simple.oracle, simple_groups).draw("mpl", fold=40), "k3-simple-logical")
display(
    pd.DataFrame(
        {
            "XAG": dict(xag.oracle.count_ops()),
            "Belletti-Simple": dict(simple.oracle.count_ops()),
        }
    )
    .fillna(0)
    .astype(int)
)

### The expanded oracles

`expand_oracle` applies the study's policy. Belletti-Simple's compute block
(the comparators) is expanded with the throwaway tree and given relative-phase
Toffoli gates, because its exact inverse follows the kick-back gate; the
conjunction on `out` is expanded with the restoring tree. The matched-pair rule
then gives relative phases to any remaining Toffoli pair. The XAG oracle has no
multi-controlled gate to expand. The appended tree ancilla becomes `a[0]`, and both
circuits now have the same width and the same Toffoli-type gates.

In [ ]:
xag_expanded, _ = expanded(xag)
simple_expanded, simple_records = expanded(simple)
display(pd.DataFrame(simple_records)[["mcx_method", "mcx_gates", "max_controls", "appended_ancillas", "garbage_ancillas"]])
xag_groups = role_groups(xag, xag_expanded)
simple_groups = role_groups(simple, simple_expanded)
show_figure(labelled(xag_expanded, xag_groups).draw("mpl", fold=40), "k3-xag-expanded")
show_figure(labelled(simple_expanded, simple_groups).draw("mpl", fold=40), "k3-simple-expanded")


def oracle_costs(circuit) -> dict:
    toffoli = costs.toffoli_metrics(circuit)
    lowered = costs.ft_metrics(costs.clifford_t(circuit))
    return {
        "qubits": circuit.num_qubits,
        "Toffoli-type gates": toffoli["toffoli_count"],
        "Toffoli types": toffoli["toffoli_types"],
        "T-count": lowered["t_count"],
        "T-depth": lowered["t_depth"],
        "CNOTs": lowered["cx_count"],
    }


def cost_table(columns: dict, expected: dict) -> pd.DataFrame:
    """Measured costs beside the analytical ones; blank where none is predicted."""
    expected = {k: v for k, v in expected.items() if k != "AND gates"}
    table = pd.concat(
        {name: pd.Series(values, dtype=object) for name, values in {**columns, "predicted": expected}.items()},
        axis=1,
    )
    table = table.loc[list(next(iter(columns.values())))].fillna("")
    for name in columns:
        for row in expected:
            assert table.loc[row, name] == expected[row], (name, row)
    return table


bits = max(1, (problem.colors - 1).bit_length())
k3_costs = cost_table(
    {"XAG": oracle_costs(xag_expanded), "Belletti-Simple": oracle_costs(simple_expanded)},
    predicted(len(problem.vertices), len(problem.edges), bits),
)
display(k3_costs)

### The exact gates that are counted

Each Toffoli-type box is drawn as the Clifford+T gadget the cost model counts,
between labelled barriers: a doubly controlled $R_x(\pm\pi)$ and a relative-phase
Toffoli gate cost 4 T gates each, and the exact Toffoli gate on `out` costs 7.

In [ ]:
show_blockwise(labelled(xag_expanded, xag_groups), "k3-xag-expanded")
show_blockwise(labelled(simple_expanded, simple_groups), "k3-simple-expanded")

## 4. Reading both circuits back as AND networks

Equal counts do not by themselves show that the circuits compute the same thing.
To compare their structure, each circuit is replayed on symbolic values. Every wire
holds a XOR of terms (the constant 1, data bits $x_{v,i}$ and AND nodes); X and
CNOT gates only rearrange these terms, so they are free. A Toffoli-type gate, with
its open controls negated, XORs the AND of its two control signals into its
target. The AND is identified by its pair of input signals, so two circuits that
compute the AND of the same signals produce the same node. A gate whose target
already holds that node uncomputes it.

The tables list every AND each circuit computes, with its inputs and what the
written qubit holds afterwards. A name $g_i$ refers to an earlier AND, and an input
such as $\neg(x_{0,0} \oplus x_{1,0})$ is an affine function of the data computed in
place by CNOT and X gates. Nodes that appear in both circuits keep the XAG's name,
so identical names mean identical gates.

In [ ]:
ONE = ("1",)


def read_back(circuit, instance):
    """Replay a circuit on symbolic XOR terms and list the AND gates it applies."""
    names = {
        instance.data[i]: f"x{vertex},{bit}"
        for vertex, inputs in instance.encoding.input_mapping.items()
        for bit, i in enumerate(inputs)
    }
    signal = {
        q: frozenset([("x", names[q])]) if q in names else frozenset()
        for q in range(circuit.num_qubits)
    }
    events = []
    for inst in circuit.data:
        operation = inst.operation
        qubits = [circuit.find_bit(q).index for q in inst.qubits]
        if operation.name == "x":
            signal[qubits[0]] ^= {ONE}
            continue
        if operation.name in ("rccx", "rccx_dg"):
            controls, state = 2, 0b11
        elif isinstance(operation, ControlledGate) and operation.base_gate.name in ("x", "rx"):
            controls, state = operation.num_ctrl_qubits, operation.ctrl_state
        else:
            raise ValueError(f"Cannot read back {operation.name}")
        inputs = [
            signal[q] ^ (frozenset() if (state >> i) & 1 else {ONE})
            for i, q in enumerate(qubits[:controls])
        ]
        target = qubits[controls]
        if controls == 1:
            signal[target] ^= inputs[0]
            continue
        node = ("AND", frozenset(inputs))
        kind = "uncompute" if node in signal[target] else "compute"
        signal[target] ^= {node}
        events.append({"node": node, "target": target, "kind": kind, "after": signal[target]})
    return events


def describe(signal, labels) -> str:
    terms = sorted(labels[t] if t[0] == "AND" else t[1] for t in signal if t != ONE)
    text = " ⊕ ".join(terms) if terms else "0"
    if ONE not in signal:
        return text
    return f"¬{text}" if len(terms) == 1 else f"¬({text})"


def name_nodes(events, labels, prefix):
    """Name the computed nodes that do not have a name yet."""
    fresh = 0
    for event in events:
        if event["kind"] == "compute" and event["node"] not in labels:
            fresh += 1
            labels[event["node"]] = f"{prefix}{fresh}"
    return labels


def and_table(events, labels, wires):
    rows = []
    for event in events:
        if event["kind"] != "compute":
            continue
        left, right = sorted(describe(s, labels) for s in event["node"][1])
        rows.append(
            {
                "AND": labels[event["node"]],
                "input 1": left,
                "input 2": right,
                "written to": wires[event["target"]],
                "qubit holds": describe(event["after"], labels),
            }
        )
    return pd.DataFrame(rows)


def computed(events):
    return {e["node"] for e in events if e["kind"] == "compute"}


xag_events = read_back(xag_expanded, xag)
simple_events = read_back(simple_expanded, simple)
labels = name_nodes(xag_events, {}, "g")
labels = name_nodes(simple_events, labels, "h")
display(Markdown("**XAG oracle**"))
display(and_table(xag_events, labels, wire_names(xag_groups)))
display(Markdown("**Belletti-Simple, throwaway tree**"))
display(and_table(simple_events, labels, wire_names(simple_groups)))
same_nodes = computed(xag_events) == computed(simple_events)
uncomputed = {
    name: sum(e["kind"] == "uncompute" for e in events)
    for name, events in (("XAG", xag_events), ("Belletti-Simple", simple_events))
}
print("Same AND gates of the same signals:", same_nodes)
print("Uncomputed ANDs:", uncomputed)
assert same_nodes

The logical Belletti-Simple oracle, before decomposition, reads back as one
$c$-input AND per edge and one $|E|$-input AND. The throwaway tree splits these
multi-input ANDs into exactly the two-input ANDs that the XAG computes.

In [ ]:
logical_events = read_back(simple.oracle, simple)
print(
    "Inputs of each AND computed by the logical Belletti-Simple oracle:",
    [len(e["node"][1]) for e in logical_events if e["kind"] == "compute"],
)


def network_drawing(events, labels, wires, title):
    """Draw the computed AND gates; dashed edges with a circle are negated inputs."""
    dot = Digraph(
        graph_attr={
            "rankdir": "LR",
            "label": title,
            "labelloc": "t",
            "fontname": "Helvetica",
            "fontsize": "14",
            "bgcolor": "white",
            "nodesep": ".2",
        },
        node_attr={"fontname": "Helvetica", "fontsize": "10", "style": "filled"},
        edge_attr={"arrowsize": ".6", "color": "#69737d"},
    )
    leaves = Digraph(name="leaves", graph_attr={"rank": "source"})
    for event in events:
        if event["kind"] != "compute":
            continue
        node = event["node"]
        name = labels[node]
        dot.node(name, f"{name} AND\n{wires[event['target']]}", shape="box", fillcolor="#dcece2")
        for signal in node[1]:
            terms = signal - {ONE}
            negated = ONE in signal
            if len(terms) == 1 and next(iter(terms))[0] == "AND":
                source = labels[next(iter(terms))]
            else:
                source = describe(signal, labels)
                leaves.node(source, source, shape="plaintext", fillcolor="white")
                negated = False
            dot.edge(
                source,
                name,
                style="dashed" if negated else "solid",
                arrowhead="odot" if negated else "normal",
            )
    dot.subgraph(leaves)
    return dot


show_svg(
    network_drawing(xag_events, labels, wire_names(xag_groups), "XAG oracle read back").pipe(format="svg"),
    "k3-xag-readback",
)
show_svg(
    network_drawing(
        simple_events, labels, wire_names(simple_groups), "Belletti-Simple (throwaway tree) read back"
    ).pipe(format="svg"),
    "k3-simple-readback",
)

## 5. Both oracles act identically

Every gate of the two expanded circuits maps a computational basis state to a
basis state up to a phase, so each circuit can be simulated exactly on all basis
inputs at once, tracking the phase of each. For all $2^6$ data assignments and
both target values, each oracle must XOR the edge-only predicate into `out`,
restore every other qubit, and leave no phase. By linearity, this establishes the
same unitary action on every state whose workspace and ancillas start in
$|0\rangle$, in particular phase kick-back with `out` in $|-\rangle$. The relative
phases of the RCCX and $R_x(\pm\pi)$ gates appear during the computation and
cancel by the end.

In [ ]:
def basis_action(circuit, bits):
    """Apply a circuit to basis states given as a (qubits × states) bit matrix."""
    bits = bits.copy()
    phases = np.full(bits.shape[1], np.exp(1j * float(circuit.global_phase)))
    cache = {}
    for inst in circuit.data:
        operation = inst.operation
        key = (
            operation.name,
            tuple(float(p) for p in operation.params),
            getattr(operation, "ctrl_state", None),
            len(inst.qubits),
        )
        if key not in cache:
            matrix = Operator(operation).data
            image = np.abs(matrix).argmax(axis=0)
            factor = matrix[image, np.arange(len(image))]
            if not np.allclose(np.abs(factor), 1):
                raise ValueError(f"{operation.name} does not map basis states to basis states")
            cache[key] = (image, factor)
        image, factor = cache[key]
        qubits = [circuit.find_bit(q).index for q in inst.qubits]
        index = sum(bits[q].astype(np.int64) << i for i, q in enumerate(qubits))
        phases *= factor[index]
        new = image[index]
        for i, q in enumerate(qubits):
            bits[q] = (new >> i) & 1
    return bits, phases


def check_xor_action(circuit, instance):
    """Exhaustively check |x>|t>|0> -> |x>|t xor f(x)>|0> without residual phase."""
    encoding = instance.encoding
    n = encoding.input_count
    columns = np.arange(1 << n)
    inputs = ((columns[None, :] >> np.arange(n)[:, None]) & 1).astype(bool)
    expected = np.array([encoding.problem.is_valid(encoding.decode(int(b))) for b in columns])
    target = instance.roles["output"][0]
    for value in (False, True):
        start = np.zeros((circuit.num_qubits, columns.size), dtype=bool)
        start[instance.data] = inputs
        start[target] = value
        bits, phases = basis_action(circuit, start)
        final = start.copy()
        final[target] ^= expected
        assert np.array_equal(bits, final), "data, workspace or target differ"
        assert np.allclose(phases, 1), "a residual phase remains"
    return expected


marked = {
    name: check_xor_action(circuit, instance)
    for name, circuit, instance in (
        ("XAG, logical", xag.oracle, xag),
        ("XAG, expanded", xag_expanded, xag),
        ("Belletti-Simple, logical", simple.oracle, simple),
        ("Belletti-Simple, expanded", simple_expanded, simple),
    )
}
display(
    pd.DataFrame(
        {
            name: {"inputs checked": 2 * len(f), "marked assignments": int(f.sum())}
            for name, f in marked.items()
        }
    ).T
)
assert all(np.array_equal(f, marked["XAG, logical"]) for f in marked.values())

## 6. A wider comparator: $K_3$ with five colors

With $k = 5$ colors, $c = 3$, and each comparator of Belletti-Simple becomes a
three-control X gate. Figure 1's throwaway tree expands it into two Toffoli gates
and one garbage ancilla; the XAG computes the same equality with two AND gates and
one workspace qubit. The trees group the color bits in opposite orders, because
the Saha–Belletti package stores the most significant bit first, so the
intermediate AND of each comparator pairs different bits. The two circuits
therefore compute the same conjunctions grouped differently; the cell compares
them both node by node and after flattening each tree of AND gates into the set of
literals it combines.

In [ ]:
def flattened(node, memo):
    """The literals that a tree of AND gates combines, whatever its grouping."""
    if node not in memo:
        literals = set()
        for signal in node[1]:
            if len(signal) == 1 and next(iter(signal))[0] == "AND":
                literals |= flattened(next(iter(signal)), memo)
            else:
                literals.add(
                    frozenset(("tree", flattened(t, memo)) if t[0] == "AND" else t for t in signal)
                )
        memo[node] = frozenset(literals)
    return memo[node]


def final_formula(events, instance):
    target = instance.roles["output"][0]
    (final,) = [e["node"] for e in events if e["kind"] == "compute" and e["target"] == target]
    return flattened(final, {})


wide = replace(problem, colors=5)
wide_xag = logique_instance(wide, "xag", "R3", executable=EXECUTABLE)
wide_simple = baseline_instance(wide, "simple", "R3")
wide_xag_expanded, _ = expanded(wide_xag)
wide_simple_expanded, _ = expanded(wide_simple)
wide_xag_groups = role_groups(wide_xag, wide_xag_expanded)
wide_simple_groups = role_groups(wide_simple, wide_simple_expanded)
show_figure(labelled(wide_simple.oracle, role_groups(wide_simple, wide_simple.oracle)).draw("mpl", fold=50), "k3c3-simple-logical")
show_figure(labelled(wide_xag_expanded, wide_xag_groups).draw("mpl", fold=50), "k3c3-xag-expanded")
show_figure(labelled(wide_simple_expanded, wide_simple_groups).draw("mpl", fold=50), "k3c3-simple-expanded")
wide_costs = cost_table(
    {"XAG": oracle_costs(wide_xag_expanded), "Belletti-Simple": oracle_costs(wide_simple_expanded)},
    predicted(3, 3, 3),
)
display(wide_costs)

In [ ]:
wide_xag_events = read_back(wide_xag_expanded, wide_xag)
wide_simple_events = read_back(wide_simple_expanded, wide_simple)
wide_labels = name_nodes(wide_xag_events, {}, "g")
wide_labels = name_nodes(wide_simple_events, wide_labels, "h")
display(Markdown("**XAG oracle**"))
display(and_table(wide_xag_events, wide_labels, wire_names(wide_xag_groups)))
display(Markdown("**Belletti-Simple, throwaway tree**"))
display(and_table(wide_simple_events, wide_labels, wire_names(wide_simple_groups)))
shared = computed(wide_xag_events) & computed(wide_simple_events)
same_formula = final_formula(wide_xag_events, wide_xag) == final_formula(wide_simple_events, wide_simple)
print(f"AND gates: {len(computed(wide_xag_events))} and {len(computed(wide_simple_events))}; shared: {len(shared)}")
print("Same flattened formula on out:", same_formula)
assert same_formula
for name, circuit, instance in (
    ("XAG", wide_xag_expanded, wide_xag),
    ("Belletti-Simple", wide_simple_expanded, wide_simple),
):
    f = check_xor_action(circuit, instance)
    print(f"{name}: exact XOR action on all {2 * len(f)} inputs, {int(f.sum())} marked")

## 7. The nine benchmark graphs

The same comparison is repeated on the nine graphs of the manuscript, without
precolors, under R3. Each instance is verified by the library as in the study
(exhaustively up to 12 inputs, otherwise on proper colorings, edge violations and
random assignments). Three constructions are compared: the XAG oracle as the
study builds it; the XAG oracle built from balanced conjunction trees, an
optimization profile of the study; and Belletti-Simple with the throwaway tree.
For each, the table gives the computed AND gates, the expanded oracle's costs,
and whether the formula written to `out` equals Belletti-Simple's once each AND
tree is flattened.

In [ ]:
NAMES = ["K3", "K5", "K10", "K20", "myciel3", "myciel4", "myciel5", "2-Insertions_3", "zed-city_2019"]
POLICIES = ["throwaway_tree", "clean_tree", "2_clean_kg24"]

rows, policy_rows, r2_rows, problems_k = [], [], [], {}
for graph in NAMES:
    instance_problem = replace(read_dimacs(GRAPHS / f"{graph}.col"), precolored={})
    problems_k[graph] = instance_problem.colors
    bits = max(1, (instance_problem.colors - 1).bit_length())
    shape = {
        "graph": graph,
        "|V|": len(instance_problem.vertices),
        "|E|": len(instance_problem.edges),
        "k": instance_problem.colors,
        "c": bits,
    }
    expected = predicted(shape["|V|"], shape["|E|"], bits)
    instances = {
        "XAG": logique_instance(instance_problem, "xag", "R3", executable=EXECUTABLE),
        "XAG, balanced": logique_instance(
            instance_problem, "xag", "R3", executable=EXECUTABLE, balanced=True
        ),
        "Belletti-Simple": baseline_instance(instance_problem, "simple", "R3"),
    }
    results = {}
    for label, instance in instances.items():
        circuit, _ = expanded(instance)
        events = read_back(circuit, instance)
        lowered = costs.ft_metrics(costs.clifford_t(circuit))
        results[label] = {
            "AND gates": len(computed(events)),
            "qubits": circuit.num_qubits,
            "T-count": lowered["t_count"],
            "T-depth": lowered["t_depth"],
            "CNOTs": lowered["cx_count"],
            "validation": instance.validation["status"],
            "formula": final_formula(events, instance),
        }
    reference = results["Belletti-Simple"]["formula"]
    for label, result in results.items():
        formula = result.pop("formula")
        rows.append(
            {
                **shape,
                "construction": label,
                **result,
                "same formula": formula == reference,
                "predicted AND gates": expected["AND gates"],
                "predicted qubits": expected["qubits"],
                "predicted T-count": expected["T-count"],
            }
        )
    for policy in POLICIES:
        circuit, _ = expanded(instances["Belletti-Simple"], policy)
        policy_rows.append(
            {
                **shape,
                "policy": policy,
                "qubits": circuit.num_qubits,
                "T-count": costs.ft_counts(circuit)["t_count"],
                "XAG qubits": results["XAG"]["qubits"],
                "XAG T-count": results["XAG"]["T-count"],
            }
        )
    for label, instance in (
        ("XAG", logique_instance(instance_problem, "xag", "R2", executable=EXECUTABLE)),
        ("Belletti-Simple", baseline_instance(instance_problem, "simple", "R2")),
    ):
        circuit, _ = expanded(instance)
        r2_rows.append(
            {
                **shape,
                "construction": label,
                "qubits": circuit.num_qubits,
                "T-count": costs.ft_counts(circuit)["t_count"],
            }
        )

nine = pd.DataFrame(rows)
display(
    nine.set_index(["graph", "construction"])[
        ["|V|", "|E|", "k", "c", "AND gates", "qubits", "T-count", "T-depth", "CNOTs", "validation", "same formula"]
    ]
)
for column in ("AND gates", "qubits", "T-count"):
    assert (nine[column] == nine[f"predicted {column}"]).all(), column
assert nine["same formula"].all()

### Comparison with the saved study

When `LOGIQUE_STUDY_RUN` names a run of `configs/corrected_comparison.json`, the
oracle costs computed here are compared with the fault-tolerant oracle rows of that
run for regime R3: the XAG oracle under every MCX policy, since it has no
multi-controlled gate, and Belletti-Simple under each policy computed above.

In [ ]:
study_run = os.environ.get("LOGIQUE_STUDY_RUN")
if study_run is None:
    display(Markdown("`LOGIQUE_STUDY_RUN` is not set; the comparison is skipped."))
else:
    summary = pd.read_csv(ROOT / study_run / "summary.csv", low_memory=False)
    saved = summary[
        (summary.component == "oracle")
        & (summary.metric_level == "fault-tolerant")
        & (summary.regime == "R3")
        & (summary.optimization_profile == "refactor")
        & summary.method.isin(["xag", "sb_simple"])
        & summary.mcx_policy.isin(POLICIES)
    ][["benchmark", "method", "mcx_policy", "qubits", "t_count"]].rename(
        columns={"qubits": "study qubits", "t_count": "study T-count"}
    )
    live = pd.concat(
        [
            pd.DataFrame(policy_rows).assign(method="sb_simple"),
            pd.DataFrame(policy_rows).assign(
                method="xag", qubits=lambda d: d["XAG qubits"], **{"T-count": lambda d: d["XAG T-count"]}
            ),
        ]
    ).rename(columns={"graph": "benchmark", "policy": "mcx_policy"})
    merged = live.merge(saved, on=["benchmark", "method", "mcx_policy"], how="left")
    merged["agrees"] = (merged["qubits"] == merged["study qubits"]) & (
        merged["T-count"] == merged["study T-count"]
    )
    display(Markdown(f"Study run `{study_run}`: {int(merged['agrees'].sum())} of {len(merged)} rows agree."))
    if not merged["agrees"].all():
        display(merged[~merged["agrees"]])
    assert merged["agrees"].all()

## 8. Where the constructions differ

**T-depth.** The counts coincide, but the order of the gates does not. The
encoder joins the edge constraints in a left-deep chain, so the XAG's conjunction
has $|E| - 1$ AND gates in sequence. The tree that expands Belletti-Simple's
conjunction has logarithmic depth, and its comparators are scheduled so that
comparators on disjoint edges overlap. The balanced profile builds the XAG's
conjunctions as trees as well; it keeps the same AND gates, qubits and T-count and
removes most of the difference in T-depth. The remainder comes from the order of
the comparators. The horizontal axis is logarithmic.

In [ ]:
SERIES = {
    "XAG": "#2a78d6",
    "XAG, balanced": "#eb6834",
    "Belletti-Simple": "#1baf7a",
}
order = list(reversed(NAMES))
figure, ax = plt.subplots(figsize=(7, 3.6))
for offset, (label, color) in zip((0.18, 0, -0.18), SERIES.items()):
    values = nine[nine["construction"] == label].set_index("graph").loc[order, "T-depth"]
    ax.scatter(values, np.arange(len(order)) + offset, s=36, color=color, label=label, zorder=3)
ax.set_xscale("log")
ax.set_yticks(range(len(order)), order)
ax.set_xlabel("T-depth of the expanded oracle (R3, throwaway tree)")
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3)
show_figure(figure, "t-depth-by-graph")
display(
    nine.pivot(index="graph", columns="construction", values="T-depth").loc[NAMES]
)

**Width under restoring decompositions.** The XAG circuit fixes one qubit for
every AND gate. Belletti-Simple keeps its multi-controlled gates until the
decomposition step, so the same oracle can be expanded with ancillas that are
restored and reused: the restoring tree (Maslov's relative-phase construction)
reuses one set of ancillas for every comparator, and Khattar and Gidney's
construction with two clean ancillas exploits conditionally clean ancillas. The
figure gives Belletti-Simple's qubits and T-count relative to the XAG oracle under
each policy. The throwaway tree sits at 1 on both axes; the restoring policies
trade T gates for qubits.

In [ ]:
POLICY_COLORS = {
    "throwaway_tree": "#2a78d6",
    "clean_tree": "#eb6834",
    "2_clean_kg24": "#1baf7a",
}
trade = pd.DataFrame(policy_rows)
trade["qubits relative to XAG"] = trade["qubits"] / trade["XAG qubits"]
trade["T-count relative to XAG"] = trade["T-count"] / trade["XAG T-count"]
figure, axes = plt.subplots(1, 2, figsize=(8, 3.6), sharey=True)
for ax, column in zip(axes, ("qubits relative to XAG", "T-count relative to XAG")):
    for offset, (policy, color) in zip((0.18, 0, -0.18), POLICY_COLORS.items()):
        values = trade[trade["policy"] == policy].set_index("graph").loc[order, column]
        ax.scatter(values, np.arange(len(order)) + offset, s=36, color=color, label=policy, zorder=3)
    ax.axvline(1, color="#69737d", linewidth=1)
    ax.set_xlabel(f"Belletti-Simple {column}")
    ax.grid(axis="x", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_yticks(range(len(order)), order)
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 0.98), ncol=3, title="Belletti-Simple MCX policy")
show_figure(figure, "policy-trade-off")
display(
    trade.pivot(index="graph", columns="policy", values=["qubits", "T-count"]).loc[NAMES]
)

**Domain checks.** Under R2 the oracle must also reject unused color codes. The
XAG flow synthesizes these checks together with the edge comparators, while
Belletti-Simple adds one $c$-control X gate and one constraint qubit for every
vertex and unused code. The equivalence therefore holds only for the edge-only
predicate; when $k$ is a power of two there are no unused codes and R2 coincides
with R3. Both oracles are expanded with the throwaway tree, as above.

In [ ]:
r2 = pd.DataFrame(r2_rows).pivot(index="graph", columns="construction", values=["qubits", "T-count"]).loc[NAMES]
r2[("T-count", "XAG / Belletti-Simple")] = (
    r2[("T-count", "XAG")] / r2[("T-count", "Belletti-Simple")]
).round(3)
display(r2)
unused = [g for g in NAMES if problems_k[g] & (problems_k[g] - 1)]
ratios = r2.loc[unused, ("T-count", "XAG / Belletti-Simple")]
equal = [g for g in unused if ratios[g] == 1]
saving = ratios.drop(equal)
display(
    Markdown(
        f"Of the {len(unused)} graphs with unused codes, the two constructions still coincide on "
        f"{', '.join(equal) or 'none'}; on the others the XAG oracle needs "
        f"{saving.min():.2f} to {saving.max():.2f} times Belletti-Simple's T gates."
    )
)

## 9. Consequences for the manuscript

* For the edge-only predicate, the XAG oracle and Belletti-Simple are the same
  circuit up to free linear gates, the grouping of conjunctions and gate order:
  both compute $|E|c - 1$ AND gates of the same signals and keep each AND on its
  own qubit. The structure-aware oracle planned as the second contribution is this
  construction, so it is not a contribution.
* The width advantage reported in the manuscript came from accounting. The
  baselines' throwaway allowance was summed over all gates, while the XAG circuit
  was counted by its allocated qubits. Counted at peak, the two are equal under the
  throwaway tree, and Belletti-Simple becomes narrower once its multi-controlled
  gates are expanded with restoring or conditionally clean ancillas. To trade
  T gates for qubits in the same way, the synthesized circuit would need a
  different mapping strategy, such as pebbling under a qubit budget.
* Logic synthesis can still matter where it acts on more than the edge
  comparators: the domain checks of R1 and R2, multiplicative-complexity
  reduction, balanced conjunctions for T-depth, and mapping under a qubit budget.

## 10. Record the run

The tables are saved as CSV files, and the configuration and versions are saved
in `equivalence_manifest.json`, beside the figures.

In [ ]:
from logique.benchmarks.provenance import checkout_revision, environment

k3_costs.to_csv(OUT / "k3_costs.csv")
wide_costs.to_csv(OUT / "k3_five_colors_costs.csv")
nine.to_csv(OUT / "nine_graphs.csv", index=False)
trade.to_csv(OUT / "policy_trade_off.csv", index=False)
pd.DataFrame(r2_rows).to_csv(OUT / "r2_costs.csv", index=False)
manifest = {
    "regime": "R3 (valid-color preparation, edge-only predicate); R2 for the domain-check table",
    "constructions": ["xag (refactor)", "xag (balanced conjunctions)", "sb_simple"],
    "mcx_policies": POLICIES,
    "relative_phase": "mirrored compute blocks and matched Toffoli pairs, as in the study",
    "graphs": NAMES,
    "study_run": study_run,
    "versions": environment(),
    "checkout": checkout_revision(ROOT),
}
(OUT / "equivalence_manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
print("Saved:", sorted(p.name for p in OUT.iterdir()))